# 05 Original Ensemble and Geometry Model Blending

This notebook evaluates whether the geometry-augmented ConvNeXt Tiny model complements the original three-model ensemble. It reproduces the original validation result, compares four predefined blends using ordinary or flip-averaged geometry predictions, and exports the selected candidate. Existing checkpoints are used for inference; no models are trained.

## 1 Setup, checkpoint discovery and split verification

The original three model checkpoints and saved geometry-model predictions are located. Validation image IDs, labels and the split fingerprint are checked to ensure that all predictions refer to the same held-out images.

In [1]:
from pathlib import Path
import json, hashlib, shutil
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torchvision.transforms as T
from torch.utils.data import Dataset,DataLoader
import timm
from tqdm.auto import tqdm

# Usually leave these automatic. Set explicit paths if multiple copies are attached.
BASE = None
RESULTS_ROOT = None
CHECKPOINT_OVERRIDES = {} # Keys: convnext_t_128, convnext_t_224, effnetv2_s_128
OUT=Path('/kaggle/working/ensemble_followup');OUT.mkdir(parents=True,exist_ok=True)
CLASSES=['CatEye','CatFace','CatMouth','CatNose','DogEye','DogFace','DogMouth','DogNose']
EXPECTED_SPLIT='4714f587e29d5c52f8d57346771d7d65bd72bbdaf36ddf21ca7260bb756524ad'
REGISTRY={'convnext_t_128':('convnext_tiny',128,.4),
          'convnext_t_224':('convnext_tiny',224,.4),
          'effnetv2_s_128':('tf_efficientnetv2_s',128,.2)}
assert torch.cuda.is_available(),'Enable a GPU.'
device=torch.device('cuda')

def image_folder(base,kind):
    for p in [base/kind/kind,base/kind]:
        if (p/'00001.jpg').is_file():return p
    raise FileNotFoundError(f'Missing {kind} images in {base}')

if BASE is None:
    candidates=[]
    for p in Path('/kaggle/input').rglob('train_labels.csv'):
        if not (p.parent/'sample_submission.csv').exists():continue
        try:image_folder(p.parent,'train');image_folder(p.parent,'test')
        except FileNotFoundError:continue
        candidates.append(p.parent)
    assert len(candidates)==1,f'Set BASE explicitly. Found: {candidates}'
    BASE=candidates[0]
BASE=Path(BASE)
TRAIN_DIR,TEST_DIR=image_folder(BASE,'train'),image_folder(BASE,'test')
sample=pd.read_csv(BASE/'sample_submission.csv')
labels=pd.read_csv(BASE/'train_labels.csv')
assert list(sample.columns)==['image_id','class'] and len(sample)==29443
assert sample.image_id.is_unique and labels.image_id.is_unique

if RESULTS_ROOT is None:
    current=Path('/kaggle/working/controlled_experiments')
    if (current/'geometry_128_val42_seed42/val_clean.npz').exists():
        RESULTS_ROOT=current
    else:
        candidates=[p.parent for p in Path('/kaggle/input').rglob('val_split_42.csv')
                    if (p.parent/'geometry_128_val42_seed42/val_clean.npz').exists()]
        assert len(candidates)==1,('Attach the unzipped controlled_experiments outputs, '
                                  f'or set RESULTS_ROOT. Found: {candidates}')
        RESULTS_ROOT=candidates[0]
RESULTS_ROOT=Path(RESULTS_ROOT)
GEO=RESULTS_ROOT/'geometry_128_val42_seed42'
val=pd.read_csv(RESULTS_ROOT/'val_split_42.csv')
assert len(val)==9547 and val.image_id.is_unique
val_ids=val.image_id.to_numpy()
split_hash=hashlib.sha256(val_ids.astype('<i8').tobytes()).hexdigest()
assert split_hash==EXPECTED_SPLIT,'Different split: do not mix these predictions.'
label_lookup=labels.set_index('image_id')['class']
assert val['class'].tolist()==label_lookup.loc[val_ids].tolist()
y=val['class'].map({c:i for i,c in enumerate(CLASSES)}).to_numpy()
geo_config=json.loads((GEO/'config.json').read_text())
assert geo_config['split_hash']==split_hash and not geo_config['full_data']
assert geo_config['label_sha256']==hashlib.sha256((BASE/'train_labels.csv').read_bytes()).hexdigest()

ckpts={}
for key in REGISTRY:
    if key in CHECKPOINT_OVERRIDES:matches=[Path(CHECKPOINT_OVERRIDES[key])]
    else:matches=list(Path('/kaggle/input').rglob(f'best_{key}.pth'))
    assert len(matches)==1 and matches[0].is_file(),f'Attach original models or set override for {key}: {matches}'
    ckpts[key]=matches[0]
print('Competition data:',BASE,'| geometry outputs:',RESULTS_ROOT)
print('Original checkpoints:',ckpts)


Competition data: /kaggle/input/datasets/khlafalshammari/competition | geometry outputs: /kaggle/input/datasets/khlafalshammari/controlled-experiments/controlled_experiments
Original checkpoints: {'convnext_t_128': PosixPath('/kaggle/input/datasets/khlafalshammari/eng707-models/best_convnext_t_128.pth'), 'convnext_t_224': PosixPath('/kaggle/input/datasets/khlafalshammari/eng707-models/best_convnext_t_224.pth'), 'effnetv2_s_128': PosixPath('/kaggle/input/datasets/khlafalshammari/eng707-models/best_effnetv2_s_128.pth')}


## 2 Probability checks and inference functions

Saved probabilities are checked for image order, class order and valid values. The original models retain their training-time preprocessing, and their predictions are averaged across original and horizontally flipped images.

In [2]:
def checked_npz(path,ids,need_labels=False):
    with np.load(path,allow_pickle=False) as a:
        assert a['classes'].tolist()==CLASSES
        assert np.array_equal(a['image_id'],ids)
        assert str(a['split_hash'])==EXPECTED_SPLIT
        p=a['probs'].copy()
        if need_labels:assert np.array_equal(a['labels'],y)
    assert p.shape==(len(ids),8) and np.isfinite(p).all() and (p>=0).all()
    assert np.allclose(p.sum(1),1,atol=1e-5)
    return p
geo_val={v:checked_npz(GEO/f'val_{v}.npz',val_ids,True) for v in ['clean','tta']}
assert int((geo_val['clean'].argmax(1)!=y).sum())==25

def score(p):
    return {'errors':int((p.argmax(1)!=y).sum()),
            'accuracy':float((p.argmax(1)==y).mean()),
            'nll':float(-np.log(np.clip(p[np.arange(len(y)),y],1e-12,1)).mean())}

class Images(Dataset):
    def __init__(self,ids,folder,size):
        self.ids=np.asarray(ids);self.folder=folder
        # Deliberately matches the original notebook for ALL three old models.
        # Do not replace with a newly resolved EfficientNet normalisation.
        self.tf=T.Compose([T.Resize((size,size)),T.ToTensor(),
                           T.Normalize([.485,.456,.406],[.229,.224,.225])])
    def __len__(self):return len(self.ids)
    def __getitem__(self,i):
        image_id=int(self.ids[i])
        with Image.open(self.folder/f'{image_id:05d}.jpg') as im:
            x=self.tf(im.convert('RGB'))
        return x,image_id

@torch.inference_mode()
def infer_old(key,ids,folder):
    arch,size,_=REGISTRY[key]
    model=timm.create_model(arch,pretrained=False,num_classes=8)
    state=torch.load(ckpts[key],map_location='cpu',weights_only=True)
    model.load_state_dict(state,strict=True);del state
    model=model.to(device).eval()
    loader=DataLoader(Images(ids,folder,size),batch_size=128,shuffle=False,
                      num_workers=2,pin_memory=True)
    probs=[];seen=[]
    for x,image_ids in tqdm(loader,desc=f'{key}: {folder.name}'):
        x=x.to(device,non_blocking=True)
        with torch.autocast('cuda',dtype=torch.float16):
            a=model(x);b=model(torch.flip(x,[3]))
        p=(a.float().softmax(1)+b.float().softmax(1))/2
        probs.append(p.cpu().numpy());seen.append(image_ids.numpy())
    p=np.concatenate(probs);seen=np.concatenate(seen)
    assert np.array_equal(seen,ids)
    assert np.isfinite(p).all() and np.allclose(p.sum(1),1,atol=1e-5)
    del model,loader;torch.cuda.empty_cache()
    return p

def save_probs(path,p,ids,labels=None):
    payload=dict(probs=p,image_id=ids,classes=np.array(CLASSES),split_hash=np.array(split_hash))
    if labels is not None:payload['labels']=labels
    np.savez_compressed(path,**payload)


## 3 Baseline reproduction and ensemble comparison

The original 40/40/20 ensemble must reproduce its 24 validation errors before new blends are compared. Four predefined candidates combine 25% or 50% geometry-model probabilities with the original ensemble, using either ordinary or flip-averaged geometry predictions. Selection prioritises fewer validation errors, with NLL as the tie breaker. 


In [3]:
old_val=np.zeros((len(val),8),dtype=np.float32)
for key,(_,_,weight) in REGISTRY.items():
    p=infer_old(key,val_ids,TRAIN_DIR)
    save_probs(OUT/f'old_{key}_val_tta.npz',p,val_ids,y)
    print(key,score(p))
    old_val+=weight*p
baseline=score(old_val)
print('Original 40/40/20 ensemble:',baseline)
save_probs(OUT/'original_ensemble_val_tta.npz',old_val,val_ids,y)
assert baseline['errors']==24,('Original result did not reproduce (expected 24 errors). '
                              'Stop and share this output before blending.')

rows=[dict(name='original_ensemble',geometry_weight=0,geometry_view='none',**baseline)]
candidates={}
old_pred=old_val.argmax(1)
for view in ['clean','tta']:
    rows.append(dict(name=f'geometry_{view}',geometry_weight=1,geometry_view=view,**score(geo_val[view])))
    for w in [.25,.5]:
        name=f'original_plus_geometry_{view}_w{w:g}'
        p=(1-w)*old_val+w*geo_val[view]
        pred=p.argmax(1)
        m=score(p)
        rows.append(dict(name=name,geometry_weight=w,geometry_view=view,**m,
                         fixed=int(((old_pred!=y)&(pred==y)).sum()),
                         broken=int(((old_pred==y)&(pred!=y)).sum())))
        candidates[name]=(p,w,view,m)
comparison=pd.DataFrame(rows)
comparison.to_csv(OUT/'original_geometry_blend_comparison.csv',index=False)
display(comparison)

# Only export a new submission if a prespecified blend reduces validation errors.
eligible=[k for k,v in candidates.items() if v[3]['errors']<baseline['errors']]
selected=min(eligible,key=lambda k:(candidates[k][3]['errors'],candidates[k][3]['nll'])) if eligible else None
if selected:
    print('Candidate for review:',selected,candidates[selected][3])
    save_probs(OUT/'selected_blend_val.npz',candidates[selected][0],val_ids,y)
else:
    print('No blend reduced validation errors. Keep the 0.9785 benchmark and share the comparison.')


convnext_t_128: train:   0%|          | 0/75 [00:00<?, ?it/s]

convnext_t_128 {'errors': 31, 'accuracy': 0.996752906672253, 'nll': 0.10004262626171112}


convnext_t_224: train:   0%|          | 0/75 [00:00<?, ?it/s]

convnext_t_224 {'errors': 34, 'accuracy': 0.996438671834084, 'nll': 0.10163145512342453}


effnetv2_s_128: train:   0%|          | 0/75 [00:00<?, ?it/s]

effnetv2_s_128 {'errors': 49, 'accuracy': 0.9948674976432387, 'nll': 0.11184898763895035}
Original 40/40/20 ensemble: {'errors': 24, 'accuracy': 0.9974861212946475, 'nll': 0.10100480169057846}


,name,geometry_weight,geometry_view,errors,accuracy,nll,fixed,broken
0,original_ensemble,0.00,none,24,0.997486,0.101005,NaN,NaN
1,geometry_clean,1.00,clean,25,0.997381,0.099777,NaN,NaN
2,original_plus_geometry_clean_w0.25,0.25,clean,22,0.997696,0.100242,4.0,2.0
3,original_plus_geometry_clean_w0.5,0.50,clean,17,0.998219,0.099739,10.0,3.0
4,geometry_tta,1.00,tta,25,0.997381,0.099016,NaN,NaN
5,original_plus_geometry_tta_w0.25,0.25,tta,22,0.997696,0.100117,3.0,1.0
6,original_plus_geometry_tta_w0.5,0.50,tta,20,0.997905,0.099477,7.0,3.0


Candidate for review: original_plus_geometry_clean_w0.5 {'errors': 17, 'accuracy': 0.998219335917042, 'nll': 0.09973899275064468}


## 4 Test predictions and selected submission export

The selected recipe is applied to test probabilities, and the submission is exported with its weights and prediction records. The original submission CSV was not attached in this execution, so exact reproduction of its test labels was not checked here. Candidate selection uses the reused validation split and does not guarantee a public-score improvement.


In [4]:
if selected:
    ids=sample.image_id.to_numpy()
    old_test=np.zeros((len(sample),8),dtype=np.float32)
    for key,(_,_,weight) in REGISTRY.items():
        p=infer_old(key,ids,TEST_DIR)
        save_probs(OUT/f'old_{key}_test_tta.npz',p,ids)
        old_test+=weight*p
    save_probs(OUT/'original_ensemble_test_tta.npz',old_test,ids)
    old_csvs=list(Path('/kaggle/input').rglob('submission_ensemble3_effhalf_tta.csv'))
    if len(old_csvs)==1:
        expected=pd.read_csv(old_csvs[0])
        assert expected.image_id.tolist()==ids.tolist()
        assert expected['class'].tolist()==np.array(CLASSES)[old_test.argmax(1)].tolist(), (
            'Original winning submission did not reproduce. Stop and review.')
        print('Original winning CSV reproduced exactly.')
    elif len(old_csvs)>1:
        raise AssertionError('Multiple original winning CSVs found; attach only one copy.')
    else:
        print('Original winning CSV not attached; exact test-prediction reproduction was not checked.')
    _,w,view,m=candidates[selected]
    geo_test=checked_npz(GEO/f'test_{view}.npz',ids)
    blend=(1-w)*old_test+w*geo_test
    assert np.allclose(blend.sum(1),1,atol=1e-5)
    save_probs(OUT/'selected_blend_test.npz',blend,ids)
    sub=sample.copy();sub['class']=np.array(CLASSES)[blend.argmax(1)]
    assert sub.image_id.is_unique and not sub.isna().any().any()
    sub.to_csv(OUT/'submission_original_plus_geometry.csv',index=False)
    recipe=dict(selected=selected,geometry_weight=w,geometry_view=view,
                original_weights={k:v[2] for k,v in REGISTRY.items()},
                split_hash=split_hash,validation=m,
                test_predictions_changed_vs_original=int((blend.argmax(1)!=old_test.argmax(1)).sum()))
    (OUT/'selected_recipe.json').write_text(json.dumps(recipe,indent=2))
    print('Saved candidate submission and recipe:',recipe)
print('Results folder:',OUT)


convnext_t_128: test:   0%|          | 0/231 [00:00<?, ?it/s]

convnext_t_224: test:   0%|          | 0/231 [00:00<?, ?it/s]

effnetv2_s_128: test:   0%|          | 0/231 [00:00<?, ?it/s]

Original winning CSV not attached; exact test-prediction reproduction was not checked.
Saved candidate submission and recipe: {'selected': 'original_plus_geometry_clean_w0.5', 'geometry_weight': 0.5, 'geometry_view': 'clean', 'original_weights': {'convnext_t_128': 0.4, 'convnext_t_224': 0.4, 'effnetv2_s_128': 0.2}, 'split_hash': '4714f587e29d5c52f8d57346771d7d65bd72bbdaf36ddf21ca7260bb756524ad', 'validation': {'errors': 17, 'accuracy': 0.998219335917042, 'nll': 0.09973899275064468}, 'test_predictions_changed_vs_original': 276}
Results folder: /kaggle/working/ensemble_followup


## 5 Results backup

Prediction arrays, comparison results, the selected recipe and the candidate submission are packaged into a backup ZIP to preserve the completed experiment.


In [5]:
archive=shutil.make_archive('/kaggle/working/ensemble_followup_backup','zip',
                            '/kaggle/working','ensemble_followup')
print('Download:',archive)


Download: /kaggle/working/ensemble_followup_backup.zip


## Summary

The original three-model ensemble reproduced its result of 24 errors across 9,547 validation images. The selected blend combined 50% original ensemble probabilities with 50% geometry-model probabilities without flip averaging. It achieved 99.8219% validation accuracy (17 errors), correcting 10 predictions while introducing 3 new errors.

The exported submission, `submission_original_plus_geometry.csv`, achieved a confirmed Kaggle public score of 0.9793, improving on the original ensemble’s 0.9785. Blend selection used the existing validation split, so the validation result remains a development estimate.